# Readmission Risk Classifier
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Luwam Major Kefali**

**Hilina Fissha Woreta**




This notebook trains the XGBoost readmission risk classifier on the preprocessed MIMIC-IV data. The model takes patient demographics, comorbidities, lab values, ICU features, and prior admissions as input and outputs a probability of 30-day readmission. We then calibrate those probabilities and use them to drive the Stage 3 enrollment decision.

## Setup and data loading

Loading the three splits from the preprocessing notebook output. The test set stays untouched throughout this notebook.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

random_seed = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

data_dir = "/kaggle/input/notebooks/hilinafissha16/preprocessing"

train = pd.read_parquet(f"{data_dir}/splits/train.parquet")
val   = pd.read_parquet(f"{data_dir}/splits/val.parquet")
test  = pd.read_parquet(f"{data_dir}/splits/test.parquet")

print("train:", train.shape)
print("val:  ", val.shape)
print("test: ", test.shape)

train: (279569, 43)
val:   (60391, 43)
test:  (59409, 43)


## Feature preparation

We drop columns that shouldn't be model inputs: identifiers, dates, the target variable, and the raw string versions of categorical columns. The categorical columns get replaced with numeric encodings since XGBoost only works with numbers.

We also drop `hospital_expire_flag` specifically because it leaks information. A patient who died in hospital can't be readmitted, so the model would learn to use death as a shortcut rather than learning actual readmission patterns.

In [2]:
cols_to_drop = [
    "subject_id", "hadm_id", "admittime", "dischtime",
    "readmitted_30d", "race_x_sex", "race_x_insurance",
    "race_clean", "insurance_clean", "sex", "admission_type",
    "hospital_expire_flag"
]

RACES            = sorted(train["race_clean"].unique().tolist())
RACE_TO_ENC      = {r: i for i, r in enumerate(RACES)}
INS_GROUPS       = sorted(train["insurance_clean"].unique().tolist())
INS_TO_ENC       = {g: i for i, g in enumerate(INS_GROUPS)}
ADMISSION_TYPES  = sorted(train["admission_type"].unique().tolist())
ADMISSION_TO_ENC = {a: i for i, a in enumerate(ADMISSION_TYPES)}

for df in [train, val, test]:
    df["sex_enc"]            = df["sex"].map({"Male": 0, "Female": 1}).fillna(-1)
    df["race_enc"]           = df["race_clean"].map(RACE_TO_ENC).fillna(-1).astype(int)
    df["insurance_enc"]      = df["insurance_clean"].map(INS_TO_ENC).fillna(-1).astype(int)
    df["admission_type_enc"] = df["admission_type"].map(ADMISSION_TO_ENC).fillna(-1).astype(int)

feature_cols = [c for c in train.columns if c not in cols_to_drop]
feature_cols = list(dict.fromkeys(feature_cols))
target = "readmitted_30d"

x_train = train[feature_cols]
y_train = train[target]
x_val   = val[feature_cols]
y_val   = val[target]
x_test  = test[feature_cols]
y_test  = test[target]

print("race encoding:", RACE_TO_ENC)
print("insurance encoding:", INS_TO_ENC)
print("admission type encoding:", ADMISSION_TO_ENC)
print("\nnumber of features:", len(feature_cols))
print("feature list:", feature_cols)
print("\nclass balance in train:")
print(y_train.value_counts())
print("\nreadmission rate:", round(y_train.mean() * 100, 1), "%")

race encoding: {'Asian': 0, 'Black/African American': 1, 'Hispanic/Latino': 2, 'Other/Unknown': 3, 'White': 4}
insurance encoding: {'Medicaid': 0, 'Medicare': 1, 'Other': 2, 'Private': 3}
admission type encoding: {'AMBULATORY OBSERVATION': 0, 'DIRECT EMER.': 1, 'DIRECT OBSERVATION': 2, 'ELECTIVE': 3, 'EU OBSERVATION': 4, 'EW EMER.': 5, 'OBSERVATION ADMIT': 6, 'SURGICAL SAME DAY ADMISSION': 7, 'URGENT': 8}

number of features: 35
feature list: ['age', 'los_days', 'cm_chf', 'cm_arrhythmia', 'cm_hypertension', 'cm_cpd', 'cm_diabetes', 'cm_renal_failure', 'cm_liver_disease', 'cm_cancer', 'cm_obesity', 'cm_depression', 'cm_anxiety', 'cm_alcohol_abuse', 'cm_drug_abuse', 'cm_psychosis', 'cm_coagulopathy', 'cm_aids', 'comorbidity_count', 'bicarbonate', 'bun', 'creatinine', 'glucose', 'hemoglobin', 'platelets', 'potassium', 'sodium', 'wbc', 'icu_los_total', 'n_icu_stays', 'prior_admissions_12m', 'sex_enc', 'race_enc', 'insurance_enc', 'admission_type_enc']

class balance in train:
readmitted_30

## Baseline model

Training a first version with sensible default parameters just to confirm everything works end to end and get a baseline AUROC before tuning.

`scale_pos_weight` handles the class imbalance, since about 79% of patients were not readmitted and 21% were. Without this the model would just predict 0 for everyone.

In [3]:
model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=1
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50
)

val_preds = model.predict_proba(x_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"\nbaseline validation AUROC: {val_auc:.4f}")

[0]	validation_0-auc:0.62393
[50]	validation_0-auc:0.69165
[100]	validation_0-auc:0.69576
[150]	validation_0-auc:0.69763
[200]	validation_0-auc:0.69880
[250]	validation_0-auc:0.69910
[299]	validation_0-auc:0.69946

baseline validation AUROC: 0.6995


## Feature importance

Looking at which features the model relied on most. This is important for the fairness analysis because if protected attributes like race or insurance type show up high on this list, it means the model is using them directly to make predictions.

In [4]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("top 15 most important features:")
print(importance.head(15).to_string(index=False))

top 15 most important features:
             feature  importance
prior_admissions_12m    0.305760
        cm_psychosis    0.051729
         n_icu_stays    0.043425
           cm_cancer    0.037874
          hemoglobin    0.037069
            race_enc    0.035628
  admission_type_enc    0.035331
            los_days    0.034131
       cm_depression    0.024175
       icu_los_total    0.021788
                 wbc    0.020087
       insurance_enc    0.019054
             sex_enc    0.018883
   comorbidity_count    0.018615
              sodium    0.018611


## Hyperparameter tuning

Using Optuna to search for better hyperparameters. We run 30 trials and pick the combination that gives the highest AUROC on the validation set.

The model learns only from the training set during this process. The validation set is used purely to score each trial.

In [5]:
def objective(trial):
    params = {
        "n_estimators":        trial.suggest_int("n_estimators", 200, 600),
        "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":           trial.suggest_int("max_depth", 4, 8),
        "subsample":           trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":    trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":    (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":        random_seed,
        "eval_metric":         "auc",
        "early_stopping_rounds": 20,
    }

    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    preds = m.predict_proba(x_val)[:, 1]
    return roc_auc_score(y_val, preds)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print("best AUROC:", round(study.best_value, 4))
print("best params:", study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

best AUROC: 0.7006
best params: {'n_estimators': 452, 'learning_rate': 0.020767149061311156, 'max_depth': 8, 'subsample': 0.6778072213972235, 'colsample_bytree': 0.951018377700339, 'min_child_weight': 10}


## Final model

Retraining with the best parameters found by Optuna. This is the model we carry forward into the fairness experiments.

In [6]:
best_params = study.best_params

best_model = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=1
)

best_model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50
)

val_preds = best_model.predict_proba(x_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"\nfinal validation AUROC: {val_auc:.4f}")

[0]	validation_0-auc:0.62829
[50]	validation_0-auc:0.69075
[100]	validation_0-auc:0.69422
[150]	validation_0-auc:0.69640
[200]	validation_0-auc:0.69782
[250]	validation_0-auc:0.69877
[300]	validation_0-auc:0.69928
[350]	validation_0-auc:0.69993
[400]	validation_0-auc:0.70028
[450]	validation_0-auc:0.70056
[451]	validation_0-auc:0.70056

final validation AUROC: 0.7006


## Calibration

Raw XGBoost scores are not well-calibrated probabilities. For example before calibration the model was outputting a mean predicted probability of 45% when the actual readmission rate is only 21%. Isotonic regression fixes this by mapping the raw scores to calibrated probabilities that match the actual observed rates.

This matters for Stage 3 because we're using these scores to make enrollment decisions, and the threshold we set should reflect real probabilities.

In [7]:
iso_reg = IsotonicRegression(out_of_bounds="clip")
iso_reg.fit(val_preds, y_val)

val_preds_calibrated = iso_reg.predict(val_preds)

print("before calibration:")
print("  mean predicted probability:", round(val_preds.mean(), 3))
print("  actual readmission rate:   ", round(y_val.mean(), 3))

print("\nafter calibration:")
print("  mean predicted probability:", round(val_preds_calibrated.mean(), 3))
print("  actual readmission rate:   ", round(y_val.mean(), 3))

before calibration:
  mean predicted probability: 0.455
  actual readmission rate:    0.211

after calibration:
  mean predicted probability: 0.211
  actual readmission rate:    0.211


## Expected Calibration Error (ECE) per Demographic Group

**Expected Calibration Error (ECE)** measures how well a model's predicted probabilities match the actual outcomes.

- A well-calibrated model that predicts **0.7** should be correct approximately **70% of the time**.
- We compute ECE separately for each **racial group** to check whether the model's calibration quality is consistent across groups.

In [8]:
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return round(ece / len(y_true), 4)


val_ece_overall = compute_ece(y_val.values, val_preds_calibrated)

ece_rows = []
for group in val["race_clean"].unique():
    mask = val["race_clean"].values == group
    ece_g = compute_ece(y_val.values[mask], val_preds_calibrated[mask])
    ece_rows.append({"group": group, "n": mask.sum(), "ece": ece_g})

ece_df = pd.DataFrame(ece_rows).sort_values("ece", ascending=False)
print(f"overall ECE: {val_ece_overall}")
print("\nECE by race (lower = better calibrated):")
print(ece_df.to_string(index=False))
print("\nTarget: ECE < 0.05 per group")

ece_df.to_parquet("/kaggle/working/stage2_ece_by_group.parquet", index=False)
print("\nECE saved to stage2_ece_by_group.parquet")


overall ECE: 0.0

ECE by race (lower = better calibrated):
                 group     n    ece
       Hispanic/Latino  3097 0.0160
         Other/Unknown  4864 0.0131
                 Asian  2008 0.0129
Black/African American  9102 0.0118
                 White 41320 0.0023

Target: ECE < 0.05 per group

ECE saved to stage2_ece_by_group.parquet


## Saving the model and predictions

Saving the trained model and calibrator so Stage 3 and the fairness experiments can load them without retraining.

In [9]:
import json
with open("/kaggle/working/stage2_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("/kaggle/working/stage2_calibrator.pkl", "wb") as f:
    pickle.dump(iso_reg, f)
with open("/kaggle/working/stage2_best_params.json", "w") as f:
    json.dump(best_params, f)

with open("/kaggle/working/stage2_encodings.json", "w") as f:
    json.dump({"race": RACE_TO_ENC, "insurance": INS_TO_ENC, "admission_type": ADMISSION_TO_ENC}, f)
    

with open("/kaggle/working/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

for split_name, split_df, x_split, y_split in [
    ("train", train, x_train, y_train),
    ("val",   val,   x_val,   y_val),
    ("test",  test,  x_test,  y_test),
]:
    raw_preds = best_model.predict_proba(x_split)[:, 1]
    cal_preds = iso_reg.predict(raw_preds)

    out = split_df[["hadm_id", "race_clean", "readmitted_30d"]].copy()
    out["y_pred"]            = raw_preds
    out["y_pred_calibrated"] = cal_preds
    out["split"]             = split_name

    out.to_parquet(f"/kaggle/working/{split_name}_predictions.parquet", index=False)
    print(f"saved {split_name}_predictions.parquet — {len(out)} rows")

print("saved stage2_model.pkl")
print("saved stage2_calibrator.pkl")
print("saved feature_cols.json")

saved train_predictions.parquet — 279569 rows
saved val_predictions.parquet — 60391 rows
saved test_predictions.parquet — 59409 rows
saved stage2_model.pkl
saved stage2_calibrator.pkl
saved feature_cols.json


## SHAP feature importance

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for every individual prediction. Positive values push the score toward readmission, negative values push it away. This satisfies EU AI Act Article 13 interpretability requirements and lets us verify how much `race_enc` is driving individual decisions.


In [10]:
!pip install shap -q
import shap

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(x_val)

shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df["hadm_id"]    = val["hadm_id"].values
shap_df["race_clean"] = val["race_clean"].values
shap_df.to_parquet("/kaggle/working/stage2_shap.parquet", index=False)
print(f"SHAP values saved: {shap_df.shape}")

mean_abs_shap = pd.DataFrame({
    "feature":    feature_cols,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("\ntop 15 features by mean |SHAP|:")
print(mean_abs_shap.head(15).to_string(index=False))

race_shap = mean_abs_shap[mean_abs_shap["feature"] == "race_enc"]
race_rank = mean_abs_shap["feature"].tolist().index("race_enc") + 1
print(f"\nrace_enc mean |SHAP|: {race_shap['mean_abs_shap'].values[0]:.4f}")
print(f"race_enc rank: {race_rank} of {len(feature_cols)} features")

race_enc_idx = feature_cols.index("race_enc")
shap_race_col = shap_values[:, race_enc_idx]
shap_by_group = (
    val[["race_clean"]].copy()
    .assign(race_enc_shap=shap_race_col)
    .groupby("race_clean")["race_enc_shap"].mean()
    .round(4).reset_index()
    .sort_values("race_enc_shap", ascending=False)
)
print("\nmean race_enc SHAP by group (positive = pushed toward enrollment):")
print(shap_by_group.to_string(index=False))

SHAP values saved: (60391, 37)

top 15 features by mean |SHAP|:
             feature  mean_abs_shap
prior_admissions_12m       0.375578
            los_days       0.164558
          hemoglobin       0.121710
  admission_type_enc       0.078260
              sodium       0.063937
                 age       0.062469
       icu_los_total       0.061026
            race_enc       0.051472
           platelets       0.044130
                 wbc       0.040513
         n_icu_stays       0.037829
                 bun       0.035467
             sex_enc       0.033717
             glucose       0.029189
         bicarbonate       0.027360

race_enc mean |SHAP|: 0.0515
race_enc rank: 8 of 35 features

mean race_enc SHAP by group (positive = pushed toward enrollment):
            race_clean  race_enc_shap
                 White         0.0320
                 Asian        -0.0075
Black/African American        -0.0146
       Hispanic/Latino        -0.0223
         Other/Unknown        -0.2800


Checking whether the race_enc SHAP direction lines up with the features actually driving the model.
`prior_admissions_12m`, `comorbidity_count`, and `n_icu_stays` are the top three features by importance, so if those differ a lot by race, that could explain why race_enc's own SHAP contribution points a different way than the raw enrollment rate by race does.

In [11]:
diagnostic_cols = ["prior_admissions_12m", "comorbidity_count", "n_icu_stays", "los_days", "hemoglobin"]
print(val.groupby("race_clean")[diagnostic_cols].mean().round(3).to_string())

                        prior_admissions_12m  comorbidity_count  n_icu_stays  los_days  hemoglobin
race_clean                                                                                        
Asian                                  1.159              1.981        0.220     5.921      10.940
Black/African American                 1.764              2.883        0.182     5.491      10.701
Hispanic/Latino                        1.505              2.445        0.173     5.316      11.154
Other/Unknown                          0.656              2.469        0.458     7.425      10.791
White                                  1.217              2.535        0.232     5.822      10.952


## Care management enrollment

This stage applies a threshold to the calibrated risk scores and enrolls the top 10% of patients into the care management program. This directly mirrors the setting from Obermeyer et al. 2019.

We then look at enrollment rates broken down by race. If the rates differ significantly across groups, that's evidence of bias in the pipeline.

In [12]:
threshold = pd.Series(val_preds_calibrated).quantile(0.90)
enrolled  = (val_preds_calibrated >= threshold).astype(int)

print("enrollment threshold:", round(threshold, 3))
print("overall enrollment rate:", round(enrolled.mean() * 100, 1), "%")

val_with_race = val[["race_clean", "hadm_id"]].copy()
val_with_race["enrolled"]   = enrolled
val_with_race["risk_score"] = val_preds_calibrated
val_with_race["y_true"]     = y_val.values

print("\nenrollment rate by race:")
print(val_with_race.groupby("race_clean")["enrolled"].mean().round(3).sort_values(ascending=False).to_string())

enrollment threshold: 0.402
overall enrollment rate: 10.2 %

enrollment rate by race:
race_clean
Black/African American    0.140
Hispanic/Latino           0.123
Asian                     0.100
White                     0.098
Other/Unknown             0.052


The enrollment rule is a **risk threshold**, not a fixed quota: any patient with a calibrated
30-day readmission risk at or above the 90th percentile qualifies for care management. This
mirrors how real hospital programs and Obermeyer et al. (2019) define enrollment, by a risk
cutoff, not a fixed headcount.

The resulting enrollment rate (see above) lands close to but not exactly at 10%. This is expected:
isotonic calibration produces a step function, so many patients near the threshold share the same
calibrated score, and the rule keeps that tied group together rather than splitting it. Forcing an
exact 10% would mean arbitrarily excluding some patients the model considers equally high-risk.

## Saving results

Saving the enrollment decisions alongside race, true labels, and risk scores. This is the baseline fairness measurement that all the mitigation methods will be compared against.

In [13]:
val_with_race.to_parquet("/kaggle/working/stage3_results.parquet", index=False)

print("results saved")
print("\nsummary:")
print("  patients evaluated:", len(val_with_race))
print("  patients enrolled: ", enrolled.sum())
print("  enrollment rate:   ", round(enrolled.mean() * 100, 1), "%")
print("\nenrollment by race:")
print(val_with_race.groupby("race_clean")["enrolled"].agg(["sum", "mean"]).round(3).to_string())

results saved

summary:
  patients evaluated: 60391
  patients enrolled:  6166
  enrollment rate:    10.2 %

enrollment by race:
                         sum   mean
race_clean                         
Asian                    200  0.100
Black/African American  1278  0.140
Hispanic/Latino          380  0.123
Other/Unknown            252  0.052
White                   4056  0.098
